### RAG Pipeline - Data Ingestion to Vector DB Pipeline

In [19]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


In [20]:
### Read all the pdfs inside the directory
def process_all_pdfs(pdf_directory):
    """Processes all PDF files in the specified directory and returns a list of documents."""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files in the directory
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process.")

    for pdf_file in pdf_files:
        print(f"Processing file: {pdf_file}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # add source information to metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = 'pdf'

            all_documents.extend(documents)
            print(f"Successfully processed {pdf_file}, extracted {len(documents)} documents.")

        except Exception as e:
            print(f"Error processing file {pdf_file}: {e}")
    return all_documents

# Process all PDFs in the specified directory
all_pdf_documents = process_all_pdfs("../data")

Found 10 PDF files to process.
Processing file: ..\data\pdf\Unacademy-Algorithms.pdf
Successfully processed ..\data\pdf\Unacademy-Algorithms.pdf, extracted 92 documents.
Processing file: ..\data\pdf\Unacademy-COA.pdf
Successfully processed ..\data\pdf\Unacademy-COA.pdf, extracted 247 documents.
Processing file: ..\data\pdf\Unacademy-Compiler Design.pdf
Successfully processed ..\data\pdf\Unacademy-Compiler Design.pdf, extracted 128 documents.
Processing file: ..\data\pdf\Unacademy-Computer Networks.pdf
Successfully processed ..\data\pdf\Unacademy-Computer Networks.pdf, extracted 193 documents.
Processing file: ..\data\pdf\Unacademy-DBMS.pdf
Successfully processed ..\data\pdf\Unacademy-DBMS.pdf, extracted 231 documents.
Processing file: ..\data\pdf\Unacademy-Digital Logic.pdf
Successfully processed ..\data\pdf\Unacademy-Digital Logic.pdf, extracted 199 documents.
Processing file: ..\data\pdf\Unacademy-Discrete Maths.pdf
Successfully processed ..\data\pdf\Unacademy-Discrete Maths.pdf, ext

In [21]:
all_pdf_documents

[Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2024-08-21T17:04:20+00:00', 'source': '..\\data\\pdf\\Unacademy-Algorithms.pdf', 'total_pages': 92, 'page': 0, 'page_label': '1', 'source_file': 'Unacademy-Algorithms.pdf', 'file_type': 'pdf'}, page_content='<Ch_Name> Asymptotic sufficiently AnalysisPB 5\n1 Asymptotic sufficiently \nAnalysis\nIntroduction\n y An Algorithm is a combination of a \nsequence of finite steps to solve a problem. \nAll steps should be finite and compulsory \n(If you are removing any one step, then the \nalgorithm will be affected).\n y Which algorithm is best suitable for a \ngiven application depends on the input \nsize, the architecture of the computer, \nand the kind of storage to be used: disks, \nmain memory, or even tapes.\n y It should terminate after a finite time and \nproduce one or more outputs by taking \nzero or more inputs.\nNeed For Analysis\n y Problem may have many candidate \nsolution, the majorit

In [22]:
### Text Splitter get into chunks
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Splits documents into smaller chunks using RecursiveCharacterTextSplitter."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, 
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")

    # show example of the first chunk
    if split_docs:
        print(f"Example of the first chunk:\n{split_docs[0].page_content[:200]}...")
        print(f"Metadata of the first chunk:\n{split_docs[0].metadata}")

    return split_docs

In [23]:
chunks = split_documents(all_pdf_documents)
chunks

Split 1879 documents into 2987 chunks.
Example of the first chunk:
<Ch_Name> Asymptotic sufficiently AnalysisPB 5
1 Asymptotic sufficiently 
Analysis
Introduction
 y An Algorithm is a combination of a 
sequence of finite steps to solve a problem. 
All steps should be...
Metadata of the first chunk:
{'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2024-08-21T17:04:20+00:00', 'source': '..\\data\\pdf\\Unacademy-Algorithms.pdf', 'total_pages': 92, 'page': 0, 'page_label': '1', 'source_file': 'Unacademy-Algorithms.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2024-08-21T17:04:20+00:00', 'source': '..\\data\\pdf\\Unacademy-Algorithms.pdf', 'total_pages': 92, 'page': 0, 'page_label': '1', 'source_file': 'Unacademy-Algorithms.pdf', 'file_type': 'pdf'}, page_content='<Ch_Name> Asymptotic sufficiently AnalysisPB 5\n1 Asymptotic sufficiently \nAnalysis\nIntroduction\n y An Algorithm is a combination of a \nsequence of finite steps to solve a problem. \nAll steps should be finite and compulsory \n(If you are removing any one step, then the \nalgorithm will be affected).\n y Which algorithm is best suitable for a \ngiven application depends on the input \nsize, the architecture of the computer, \nand the kind of storage to be used: disks, \nmain memory, or even tapes.\n y It should terminate after a finite time and \nproduce one or more outputs by taking \nzero or more inputs.\nNeed For Analysis\n y Problem may have many candidate \nsolution, the majorit

### Embedding and VectorStoreDB

In [24]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [25]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initializes the embedding manager.
        
        Args:
            model_name (str): The name of the sentence transformer model to use for generating embeddings.

        """

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Loads the sentence transformer model."""
        try:
            print(f"Loading embedding model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Successfully loaded embedding model: {self.model_name} with embedding dimension {self.model.get_sentence_embedding_dimension()}.")
        except Exception as e:
            print(f"Error loading embedding model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generates embeddings for a list of texts.
        
        Args:
            texts (List[str]): A list of strings to generate embeddings for.

        Returns:
            np.ndarray: An array of embeddings for the input texts.
        """

        if not self.model:
            raise ValueError("Embedding model is not loaded. Please check the model name and try again.")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Finished generating embeddings: {embeddings.shape}")
        return embeddings

## intialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2136.42it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Successfully loaded embedding model: all-MiniLM-L6-v2 with embedding dimension 384.


### VectorStore

In [26]:
class VectorStore:
    """A simple vector store implementation using ChromaDB to store and retrieve document embeddings."""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/chromadb_data"):
        """Initializes the vector store.
        
        Args:
            collection_name (str): The name of the ChromaDB collection to use for storing embeddings.
            persist_directory (str): The directory to persist the ChromaDB data.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()


    def _initialize_store(self):
        """Initializes the ChromaDB client and collection."""
        try:
            # create persist directory if it doesn't exist
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # get or create collection
            self.collection = self.client.get_or_create_collection(name=self.collection_name, metadata={"description": "PDF document embeddings for RAG"})

            print(f"Successfully initialized ChromaDB collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Adds documents and their corresponding embeddings to the vector store.
        
        Args:
            documents (List[Any]): A list of document objects to add to the store. Each document should have a unique identifier in its metadata.
            embeddings (np.ndarray): An array of embeddings corresponding to the input documents.
        """
        if not self.collection:
            raise ValueError("Vector store collection is not initialized. Please check the initialization and try again.")

        if len(documents) != len(embeddings):
            raise ValueError("The number of documents must match the number of embeddings.")

        # Prepare data for insertion
        ids = []  # Generate unique IDs for each document
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"  # Generate a unique ID for the document
            ids.append(doc_id)

            # Create a copy of the document's metadata and add additional fields
            metadata = doc.metadata.copy()
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            
            metadatas.append(metadata)
            
            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())  # Convert numpy array to list for JSON serialization

        # add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text,
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Total documents in collection after addition: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vector_store = VectorStore()
vector_store

Successfully initialized ChromaDB collection: pdf_documents
Existing documents in collection: 12


In [27]:
chunks

[Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2024-08-21T17:04:20+00:00', 'source': '..\\data\\pdf\\Unacademy-Algorithms.pdf', 'total_pages': 92, 'page': 0, 'page_label': '1', 'source_file': 'Unacademy-Algorithms.pdf', 'file_type': 'pdf'}, page_content='<Ch_Name> Asymptotic sufficiently AnalysisPB 5\n1 Asymptotic sufficiently \nAnalysis\nIntroduction\n y An Algorithm is a combination of a \nsequence of finite steps to solve a problem. \nAll steps should be finite and compulsory \n(If you are removing any one step, then the \nalgorithm will be affected).\n y Which algorithm is best suitable for a \ngiven application depends on the input \nsize, the architecture of the computer, \nand the kind of storage to be used: disks, \nmain memory, or even tapes.\n y It should terminate after a finite time and \nproduce one or more outputs by taking \nzero or more inputs.\nNeed For Analysis\n y Problem may have many candidate \nsolution, the majorit

In [28]:
### convert the chunks to text and generate embeddings
texts = [doc.page_content for doc in chunks]

## generate embeddings for the chunks
embeddings = embedding_manager.generate_embeddings(texts)

## store in the vector database
vector_store.add_documents(chunks, embeddings)

Generating embeddings for 2987 texts...


Batches: 100%|██████████| 94/94 [05:25<00:00,  3.47s/it]


Finished generating embeddings: (2987, 384)
Successfully added 2987 documents to the vector store.
Total documents in collection after addition: 2999


### Retriever Pipeline From VectorStore

In [38]:
class RAGRetriever:
    """A retriever class for performing similarity search on document embeddings stored in a vector store."""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """Initializes the RAG retriever.
        
        Args:
            vector_store (VectorStore): An instance of the VectorStore class to retrieve documents from.
            """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """Retrieves the most relevant documents for a given query.
        
        Args:
            query (str): The input query string to search for relevant documents.
            top_k (int): The number of top relevant documents to retrieve.
            score_threshold (float): The minimum similarity score for retrieved documents.
        Returns:
            List[Dict[str, Any]]: A list of dictionaries containing the retrieved documents and their metadata.
        """

        print(f"Retrieving documents for query: '{query}' with top_k={top_k} and score_threshold={score_threshold}...")

        # generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # search in the vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
                )

            # Process results and filter by score threshold
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:  # Check if there are any results
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance  # Convert distance to similarity score

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })
                        
                print(f"Retrieved {len(retrieved_docs)} documents after filtering.")
            else:
                print("No documents retrieved for the query.")

            return retrieved_docs
        
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
        
rag_retriever = RAGRetriever(vector_store, embedding_manager)


In [39]:
rag_retriever


In [49]:
rag_retriever.retrieve("what is DBMS", top_k=3, score_threshold=0.2)

Retrieving documents for query: 'what is DBMS' with top_k=3 and score_threshold=0.2...
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 31.56it/s]

Finished generating embeddings: (1, 384)
Retrieved 2 documents after filtering.


[{'id': 'doc_4d43b284_1002',
  'content': 'Introduction to DBMS\n Chapter 1\n1\n1. INTRODUCTION TO DBMS\nA system which manages all such inter-related data is called “Database \nManagement System”. It plays a vital role in almost all areas where \ncomputers are used.\nFor example, a company can have a database of their employees, including \ntheir name, address, age, date of birth and salaries.\nDBMS: \nCollection of \ninter-related date.\nSet of software tools/programs \nthat operate on collection of data\nDatabase management system can be defined as software that helps to \nmaintain and utilize a huge amount of data.\nFig. 1.1 A Simplified Database System Environment\nIntroduction to DBMS\n1',
  'metadata': {'producer': 'iLovePDF',
   'source_file': 'Unacademy-DBMS.pdf',
   'creationdate': '',
   'doc_index': 1002,
   'moddate': '2024-08-21T16:52:32+00:00',
   'content_length': 641,
   'page': 0,
   'total_pages': 231,
   'creator': 'PyPDF',
   'page_label': '1',
   'file_type': 'pdf

In [55]:
rag_retriever.retrieve("what is Multiprogramming", top_k=3, score_threshold=0.0)


Retrieving documents for query: 'what is Multiprogramming' with top_k=3 and score_threshold=0.0...
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  6.85it/s]

Finished generating embeddings: (1, 384)
Retrieved 3 documents after filtering.


[{'id': 'doc_09e511d7_2274',
  'content': '7\nIntroduction To Operating System \n Chapter 1\nMultiprogramming:\n y Multiprogramming came into the picture to overcome the shortcomings \nof Uniprogramming.\n y Operating system is able to manage multiple ready to run programs in \nmemory at once. \nNote:\nIn a multiprogramming environment processor needs to run various process, \nso processes need context switching from CPU to memory frequently, \nbecause only one process can run on the CPU at a time.\nDisk\nP4 P5\nOperating\nSystem\nP1\nCPU\nMain\nMemory\nP2\nP3\nFig. 1.7 Multiprogramming Memory Management\nTechnique\nExample:\nMultiprogramming, Multitasking, Real-Time and Modern Operating Systems \nlike Windows 7, 8, 10.\nTypes of multiprogramming:\n1) Non-Preemptive based:\nIn this case, the process currently running releases the CPU voluntarily:\nThe process releases CPU: \ni) When they require I/O\nii) When the process invokes the system call\niii) When the process gets completed.',


### Integration VectorDB Context pipeline with LLM output

In [ ]:
### simple rag peipla with groq llm
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv


### initilaise the groq llm
groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(api_key=groq_api_key, model="llama-3.1-8b-instant", temperature=0.1, max_tokens=1024)


### simple RAG function: retreive + context + generate answer
def rag_simple(query, retriever, llm, top_k=3):
    ### retrieve relevant documents
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else "No relevant documents found."
    if not context:
        return "No relevant documents found to answer the query."
    
    ##generate ans using groq llm
    prompt = f"Use the following context to answer the question:\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:"
    
    response = llm.invoke([prompt.format(context=context, query=query)])

    return response.content



In [63]:
answer = rag_simple("what is DBMS?", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'what is DBMS?' with top_k=3 and score_threshold=0.0...
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 31.07it/s]

Finished generating embeddings: (1, 384)
Retrieved 3 documents after filtering.


A DBMS (Database Management System) is a collection of inter-related data and a set of software tools/programs that operate on this collection of data. It is software that helps to maintain and utilize a huge amount of data.


### Enhance RAG Pipeline Features

In [74]:
# Enhancced RAG Pipeline Features
from annotated_types import doc


def rag_advanced(query, retriever, llm, top_k=5, min_score=0.0, return_context=False):
    """
    Enhanced RAG function with additional features.
    - Returns answer, sources, confidence score, and optionally the retrieved context.
    """
    ### retrieve relevant documents
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    
    if not results:
        return {'answer': "No relevant documents found to answer the query.", 'sources': [], 'confidence_score': 0.0, 'context': "" if return_context else None}

    #prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:200]  # first 200 characters as preview
    } for doc in results]
    confidence = max(doc['similarity_score'] for doc in results)  # use the highest similarity score as confidence
    
    ##generate ans using groq llm
    prompt = f"Use the following context to answer the question:\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:"
    
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence,
    }
    if return_context:
        output['context'] = context
    return output

# Example usage of the enhanced RAG function

result = rag_advanced("What is Operating System?", rag_retriever, llm, top_k=5, min_score=0.2, return_context=True)
print("Answer:", result['answer'])
print("\nSources:", result['sources'])
print("\nConfidence:", result['confidence'])
print("\nContext Preview:", result.get('context', 'No context returned'))

Retrieving documents for query: 'What is Operating System?' with top_k=5 and score_threshold=0.2...
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.60it/s]


Finished generating embeddings: (1, 384)
Retrieved 5 documents after filtering.
Answer: An operating system is a software that acts as a mediator/interface between the user and computer hardware. It performs various tasks such as memory management, process management, file management, etc. to provide a convenient and efficient environment for users to interact with the computer hardware.

Sources: [{'source': 'Unacademy-Operating Systems.pdf', 'page': 0, 'score': 0.5659476220607758, 'preview': '1\nIntroduction To  Operating System\n Chapter 1\n1.1 OPERATING SYSTEM FUNDAMENTALS\nIntroduction:\nDefinition\n“An operating system is a software which acts as a mediator/interface \nin between the user and'}, {'source': 'Unacademy-Operating Systems.pdf', 'page': 24, 'score': 0.3800662159919739, 'preview': '25\nIntroduction To Operating System \n Chapter 1Chapter Summary\n·\t\tOperating System: It is an environment, resource allocator, interface between \nhardware and computer user.\n·\t\tGoals

In [75]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what computer architecture is", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what computer architecture is' with top_k=3 and score_threshold=0.1...
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  9.42it/s]

Finished generating embeddings: (1, 384)
Retrieved 3 documents after filtering.
Streaming answer:
Use the following context to answer the question concisely.
Context:
Basics of Computer Design
 Chapter 1
1
1.1 COMPUTER ARCHITECTURE
 y It deals with the syste

m attributes which are visible to the programmer, 
and these attributes have a direct impact on the execution of the program.
 y Few examples of such attributes are I/O mechanism, bits required to 
represent data types, instruction sets, etc.
Computer organisation:
 y In computer organisation we deal with the operational unit of the 
hardware system and its interconnections
 y Examples of organisational attributes are hardware details that are 
hidden from the programmer.
Von–Neumann architecture:
 y It is based on the concept of the same memory holding both data and 
instructions.
 y In this architecture, we have common addresses and data buses for 
both CPU and Memory.
 y John Von Neumann proposed this architecture in 1945.
 y In this architecture, single instruction takes 2 clock cycles.
 y In this architecture, CPU cannot access instructions and read/write at 
the  same interval of time.

2
Introduction To Operating System Chapter 1
Note:
Note:
We can implement any Boolean function